In [1]:
%load_ext autotime

time: 128 µs (started: 2026-09-23 21:44:00 +05:30)


In [2]:
import json
import os
import shutil
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

DATA = Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore", category=FutureWarning)

import fastapi as _fastapi, flask as _flask, pydantic as _pydantic
import sklearn as _sklearn, xgboost as _xgboost

# Defaults, APIs and error shapes all move between releases. Pin what this ran on.
print("versions        :", f"flask {_flask.__version__} · fastapi {_fastapi.__version__} "
      f"· pydantic {_pydantic.__version__} · scikit-learn {_sklearn.__version__} "
      f"· xgboost {_xgboost.__version__}")
print("data files      :", sorted(p.name for p in DATA.glob("*.csv")))
print("service files   :", sorted(p.name for p in Path(".").glob("*_app.py")))
print("artifacts so far:", sorted(p.name for p in ARTIFACTS.glob("*")) or "(empty — Part 1 fills this)")

versions        : flask 3.1.3 · fastapi 0.139.0 · pydantic 2.12.5 · scikit-learn 1.7.2 · xgboost 3.2.0
data files      : []
service files   : []
artifacts so far: (empty — Part 1 fills this)
time: 1.44 s (started: 2026-09-23 21:44:04 +05:30)


/var/folders/p6/6_nprx9x22s4njm57gzb34l40000gp/T/ipykernel_21934/700244308.py:26: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("versions        :", f"flask {_flask.__version__} · fastapi {_fastapi.__version__} "


In [3]:
cars = pd.read_csv(DATA / "cars24-car-price.csv")
print(cars.shape)
cars.head()

(19980, 11)


,full_name,selling_price,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti Alto Std,1.20,2012.0,Individual,120000,Petrol,Manual,19.70,796.0,46.30,5.0
1,Hyundai Grand i10 Asta,5.50,2016.0,Individual,20000,Petrol,Manual,18.90,1197.0,82.00,5.0
2,Hyundai i20 Asta,2.15,2010.0,Individual,60000,Petrol,Manual,17.00,1197.0,80.00,5.0
3,Maruti Alto K10 2010-2014 VXI,2.26,2012.0,Individual,37000,Petrol,Manual,20.92,998.0,67.10,5.0
4,Ford Ecosport 2015-2021 1.5 TDCi Titanium BSIV,5.70,2015.0,Dealer,30000,Diesel,Manual,22.77,1498.0,98.59,5.0


time: 31.3 ms (started: 2026-09-23 21:45:21 +05:30)


In [4]:
encode_dict = {
    "fuel_type": {"Diesel": 1, "Petrol": 2, "CNG": 3, "LPG": 4, "Electric": 5},
    "transmission_type": {"Manual": 1, "Automatic": 2},
    "seller_type": {"Dealer": 1, "Individual": 2, "Trustmark Dealer": 3},
}

PRICE_FEATURES = [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "mileage",
    "engine",
    "max_power",
    "seats",
]

df = cars.drop(columns=["full_name"]).replace(encode_dict).infer_objects(copy=False)
X, y = df[PRICE_FEATURES], df["selling_price"]
X.head()

,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,2012.0,2,120000,2,1,19.70,796.0,46.30,5.0
1,2016.0,2,20000,2,1,18.90,1197.0,82.00,5.0
2,2010.0,2,60000,2,1,17.00,1197.0,80.00,5.0
3,2012.0,2,37000,2,1,20.92,998.0,67.10,5.0
4,2015.0,1,30000,1,1,22.77,1498.0,98.59,5.0


time: 19.7 ms (started: 2026-09-23 21:45:43 +05:30)


In [5]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

BEST_PARAMS = {
    "n_estimators": 404,
    "max_depth": 7,
    "learning_rate": 0.06607192356835712,
    "subsample": 0.994398722227957,
    "colsample_bytree": 0.6118312727750044,
    "reg_lambda": 0.2758980370254515,
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

price_model = XGBRegressor(random_state=42, **BEST_PARAMS).fit(X_train, y_train)

print(f"test MAE : {mean_absolute_error(y_test, price_model.predict(X_test)):.4f} lakhs   <- goes into model_meta.json")

test MAE : 0.9815 lakhs   <- goes into model_meta.json
time: 1.17 s (started: 2026-09-23 21:46:04 +05:30)


In [6]:
loans = pd.read_csv(DATA / "train_flask.csv")
print(loans.shape)
loans.head()

(614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


time: 5.47 ms (started: 2026-09-23 21:46:35 +05:30)


In [7]:
loan_encode_dict = {
    "Gender": {"Male": 0, "Female": 1},
    "Married": {"No": 0, "Yes": 1},
    "Credit_History": {"Uncleared Debts": 0, "Cleared Debts": 1},
}

LOAN_FEATURES = ["Gender", "Married", "ApplicantIncome", "LoanAmount", "Credit_History"]

loans["Gender"] = loans["Gender"].map(loan_encode_dict["Gender"])
loans["Married"] = loans["Married"].map(loan_encode_dict["Married"])
loans["Loan_Status"] = loans["Loan_Status"].map({"N": 0, "Y": 1})

loans = loans.dropna(subset=LOAN_FEATURES + ["Loan_Status"])

X_loan, y_loan = loans[LOAN_FEATURES], loans["Loan_Status"]
print("rows after dropna:", X_loan.shape[0], "of 614")
print("approval rate    :", round(float(y_loan.mean()), 3))

rows after dropna: 529 of 614
approval rate    : 0.69
time: 3.54 ms (started: 2026-09-23 21:48:01 +05:30)


In [8]:
from sklearn.linear_model import LogisticRegression

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan
)

loan_model = LogisticRegression(max_iter=1000).fit(Xl_train, yl_train)

print("train accuracy:", round(loan_model.score(Xl_train, yl_train), 3))
print("test  accuracy:", round(loan_model.score(Xl_test, yl_test), 3))

train accuracy: 0.811
test  accuracy: 0.849
time: 83.1 ms (started: 2026-09-23 21:49:03 +05:30)


In [9]:
import sklearn

DECISION_THRESHOLD = 0.80
MODEL_VERSION = "milepost-2026-09-23"

joblib.dump(price_model, ARTIFACTS / "price_model.joblib")
joblib.dump(loan_model, ARTIFACTS / "loan_model.joblib")

(ARTIFACTS / "model_meta.json").write_text(json.dumps({
    "model_version": MODEL_VERSION,
    "decision_threshold": DECISION_THRESHOLD,
    "price_features": PRICE_FEATURES,
    "loan_features": LOAN_FEATURES,
    "price_test_mae_lakhs": round(float(mean_absolute_error(y_test, price_model.predict(X_test))), 4),
    "sklearn_version": sklearn.__version__,
}, indent=2))

for p in sorted(ARTIFACTS.glob("*")):
    print(f"  {p.name:<24} {p.stat().st_size / 1024:8.1f} KB")

  loan_model.joblib             1.3 KB
  model_meta.json               0.4 KB
  price_model.joblib         2470.8 KB
time: 11.8 ms (started: 2026-09-23 21:50:01 +05:30)


In [10]:
import socket

for host in ["localhost", "www.google.com"]:
    try:
        print(f"{host:<18} -> {socket.gethostbyname(host)}")
    except OSError as exc:
        print(f"{host:<18} -> lookup failed ({exc})")

localhost          -> 127.0.0.1
www.google.com     -> 142.251.150.119
time: 2.67 ms (started: 2026-09-23 21:53:03 +05:30)
